## Transform race_results table


### Step 0 - Parameters and Imports


In [0]:
%run "../utils/config"

In [0]:
from pyspark.sql.functions import current_timestamp


### Step 1 - Get the dfs

In [0]:
races_df = spark.read \
    .table(f"{processed_folder_path}/races")

circuits_df = spark.read \
    .format("parquet") \
    .load(f"{processed_folder_path}/circuits")


drivers_df = spark.read \
    .format("parquet") \
    .load(f"{processed_folder_path}/drivers")

constructors_df = spark.read \
    .format("parquet") \
    .load(f"{processed_folder_path}/constructors")

results_df = spark.read \
    .format("parquet") \
    .load(f"{processed_folder_path}/results")

### Step 2 - Transform data

In [0]:
race_results_df = (
    results_df
    .join(races_df, results_df['race_id'] == races_df['race_id'])
    .join(drivers_df, results_df["driver_id"] == drivers_df["driver_id"])
    .join(circuits_df, races_df["circuit_id"] == circuits_df["circuit_id"])
    .join(constructors_df, results_df['constructor_id'] == constructors_df['constructor_id'])
    .select(
        races_df["race_year"],
        races_df["name"].alias("race_name"),
        races_df["race_timestamp"].alias("race_date"),
        circuits_df["location"].alias("circuit_location"),
        drivers_df["name"].alias("driver_name"),
        drivers_df["number"].alias("driver_number"),
        drivers_df["nationality"].alias("driver_nationality"),
        constructors_df["name"].alias("team"),
        constructors_df["nationality"].alias("team_nationality"),
        results_df["position"],
        results_df["grid"],
        results_df["fastest_lap"],
        results_df["time"].alias("race_time"),
        results_df["points"],
    )
    .withColumn("ingestion_date", current_timestamp())
)

### Step 3 - Write data


In [0]:
race_results_df.write.format("delta").mode("overwrite").saveAsTable("f1_presentation.race_results")